In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# PbSO₄ — neutron powder, constant wavelength, empirical asymmetry

In [2]:
import easydiffraction as edi
from easydiffraction import ExperimentFactory
from easydiffraction import StructureFactory
from easydiffraction.analysis import verification as verify

## Build the project

In [3]:
project = edi.Project()

## Define the structure

In [4]:
structure = StructureFactory.from_scratch(name='pbso4')

structure.space_group.name_h_m = 'P n m a'  # FullProf Space group symbol

structure.cell.length_a = 8.479506  # FullProf a
structure.cell.length_b = 5.397256  # FullProf b
structure.cell.length_c = 6.958973  # FullProf c

structure.atom_sites.create(
    id='Pb',  # FullProf Atom
    type_symbol='Pb',  # FullProf Typ
    fract_x=0.18752,  # FullProf X
    fract_y=0.25,  # FullProf Y
    fract_z=0.16705,  # FullProf Z
    adp_type='Biso',  # FullProf Biso
    adp_iso=1.39017,  # FullProf Biso
)
structure.atom_sites.create(
    id='S',  # FullProf Atom
    type_symbol='S',  # FullProf Typ
    fract_x=0.06549,  # FullProf X
    fract_y=0.25,  # FullProf Y
    fract_z=0.68373,  # FullProf Z
    adp_type='Biso',  # FullProf Biso
    adp_iso=0.39270,  # FullProf Biso
)
structure.atom_sites.create(
    id='O1',  # FullProf Atom
    type_symbol='O',  # FullProf Typ
    fract_x=0.90816,  # FullProf X
    fract_y=0.25,  # FullProf Y
    fract_z=0.59544,  # FullProf Z
    adp_type='Biso',  # FullProf Biso
    adp_iso=1.99307,  # FullProf Biso
)
structure.atom_sites.create(
    id='O2',  # FullProf Atom
    type_symbol='O',  # FullProf Typ
    fract_x=0.19355,  # FullProf X
    fract_y=0.25,  # FullProf Y
    fract_z=0.54331,  # FullProf Z
    adp_type='Biso',  # FullProf Biso
    adp_iso=1.47771,  # FullProf Biso
)
structure.atom_sites.create(
    id='O3',  # FullProf Atom
    type_symbol='O',  # FullProf Typ
    fract_x=0.08109,  # FullProf X
    fract_y=0.02727,  # FullProf Y
    fract_z=0.80869,  # FullProf Z
    adp_type='Biso',  # FullProf Biso
    adp_iso=1.30007,  # FullProf Biso
)

project.structures.add(structure)

## Load the FullProf reference

In [5]:
FULLPROF_PROJECT_DIR = 'pd-neut-cwl_pv-asym_empir_pbso4'
FULLPROF_PRF_FILE = 'pbso4.prf'
FULLPROF_BAC_FILE = 'pbso4.bac'
FULLPROF_ZERO = -0.08424  # FullProf Zero
FULLPROF_SCALE = 1.463815  # FullProf Scale
FULLPROF_WAVELENGTH = 1.912000  # FullProf Lambda
FULLPROF_U = 0.153402  # FullProf U
FULLPROF_V = -0.453103  # FullProf V
FULLPROF_W = 0.419409  # FullProf W
FULLPROF_X = 0.0  # FullProf X
FULLPROF_Y = 0.086818  # FullProf Y
FULLPROF_ASY_1 = 0.29465  # FullProf Asy1
FULLPROF_ASY_2 = 0.02261  # FullProf Asy2
FULLPROF_ASY_3 = -0.10961  # FullProf Asy3
FULLPROF_ASY_4 = 0.04941  # FullProf Asy4

x, calc_fullprof = verify.load_fullprof_calc_profile(
    FULLPROF_PROJECT_DIR,
    FULLPROF_PRF_FILE,
    FULLPROF_BAC_FILE,
    FULLPROF_ZERO,
)

## Create the experiment

In [6]:
experiment = ExperimentFactory.from_scratch(
    name='pbso4',
    sample_form='powder',
    beam_mode='constant wavelength',
    radiation_probe='neutron',
    scattering_type='bragg',
)
verify.set_reference_as_measured(experiment, x, calc_fullprof)

experiment.linked_structures.create(structure_id='pbso4', scale=FULLPROF_SCALE)

experiment.instrument.setup_wavelength = FULLPROF_WAVELENGTH
experiment.instrument.calib_twotheta_offset = FULLPROF_ZERO

experiment.peak.type = 'pseudo-voigt'
experiment.peak.broad_gauss_u = FULLPROF_U
experiment.peak.broad_gauss_v = FULLPROF_V
experiment.peak.broad_gauss_w = FULLPROF_W
experiment.peak.broad_lorentz_x = FULLPROF_X
experiment.peak.broad_lorentz_y = FULLPROF_Y
# The empirical-asymmetry coefficients (cryspy only) are set in the
# cryspy section below; crysfml has no empirical-asymmetry model.

project.experiments.add(experiment)

Peak profile type for experiment 'pbso4' changed to


pseudo-voigt


## edi-cryspy VS FullProf

In [7]:
experiment.peak.type = 'pseudo-voigt + empirical asymmetry'
experiment.peak.broad_gauss_u = FULLPROF_U
experiment.peak.broad_gauss_v = FULLPROF_V
experiment.peak.broad_gauss_w = FULLPROF_W
experiment.peak.broad_lorentz_x = FULLPROF_X
experiment.peak.broad_lorentz_y = FULLPROF_Y
experiment.peak.asym_empir_1 = FULLPROF_ASY_1
experiment.peak.asym_empir_2 = FULLPROF_ASY_2
experiment.peak.asym_empir_3 = FULLPROF_ASY_3
experiment.peak.asym_empir_4 = FULLPROF_ASY_4

experiment.calculator.type = 'cryspy'

project.analysis.calculate()
calc_ed_cryspy = experiment.data.intensity_calc

project.display.pattern_comparison(
    'pbso4',
    reference=calc_fullprof,
    candidate=calc_ed_cryspy,
    reference_label='FullProf',
    candidate_label='edi-cryspy',
)

⚠️ Switching peak profile type adds these settings with defaults:                                                                 
   • asym_empir_1=0.0                                                                                                             
   • asym_empir_2=0.0                                                                                                             
   • asym_empir_3=0.0                                                                                                             
   • asym_empir_4=0.0                                                                                                             


⚠️ Switching peak profile type resets these settings to defaults:                                                                 
   • broad_gauss_u: 0.153402 -> 0.01                                                                                              
   • broad_gauss_v: -0.453103 -> -0.01                                                                                            
   • broad_gauss_w: 0.419409 -> 0.02                                                                                              
   • broad_lorentz_y: 0.086818 -> 0.0                                                                                             


Peak profile type for experiment 'pbso4' changed to


pseudo-voigt + empirical asymmetry


Calculator for experiment 'pbso4' already set to


cryspy


## Fit edi-cryspy to FullProf

In [8]:
# cryspy and FullProf parameterise the empirical asymmetry differently, so
# the FullProf coefficients do not transfer 1-to-1. Freeing cryspy's own
# coefficients recovers the FullProf profile, confirming the structure and
# the symmetric profile are correct.
experiment.calculator.type = 'cryspy'

experiment.linked_structures['pbso4'].scale.free = True
experiment.peak.asym_empir_1.free = True
experiment.peak.asym_empir_2.free = True
experiment.peak.asym_empir_3.free = True
experiment.peak.asym_empir_4.free = True

project.analysis.fit()
project.display.fit.results()

project.analysis.calculate()
calc_ed_cryspy_refined = experiment.data.intensity_calc

project.display.pattern_comparison(
    'pbso4',
    reference=calc_fullprof,
    candidate=calc_ed_cryspy_refined,
    reference_label='FullProf',
    candidate_label='edi-cryspy (refined)',
)

Calculator for experiment 'pbso4' already set to


cryspy


<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'pbso4' for 'single' fitting


...

,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,lmfit (leastsq)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),3.58
4,🔁 Iterations,19
5,📏 Goodness-of-fit (reduced χ²),7.46
6,"📏 R-factor (Rf, %)",0.67
7,"📏 R-factor squared (Rf², %)",0.91
8,"📏 Weighted R-factor (wR, %)",0.91


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,pbso4,linked_structure,pbso4,scale,,1.4638,1.4615,0.0003,0.16 % ↓
2,pbso4,peak,,asym_empir_1,,0.2947,-0.3651,0.0027,223.91 % ↓
3,pbso4,peak,,asym_empir_2,,0.0226,-0.0236,0.0005,204.36 % ↓
4,pbso4,peak,,asym_empir_3,,-0.1096,0.0284,0.0064,125.95 % ↓
5,pbso4,peak,,asym_empir_4,,0.0494,-0.0380,0.0012,176.83 % ↓


## edi-crysfml VS FullProf

In [9]:
experiment.calculator.type = 'crysfml'

experiment.linked_structures['pbso4'].scale = FULLPROF_SCALE

experiment.peak.type = 'pseudo-voigt'
experiment.peak.broad_gauss_u = FULLPROF_U
experiment.peak.broad_gauss_v = FULLPROF_V
experiment.peak.broad_gauss_w = FULLPROF_W
experiment.peak.broad_lorentz_x = FULLPROF_X
experiment.peak.broad_lorentz_y = FULLPROF_Y

project.analysis.calculate()
calc_ed_crysfml = experiment.data.intensity_calc

project.display.pattern_comparison(
    'pbso4',
    reference=calc_fullprof,
    candidate=calc_ed_crysfml,
    reference_label='FullProf',
    candidate_label='edi-crysfml',
)

Calculator for experiment 'pbso4' changed to


crysfml


⚠️ Switching peak profile type removes these settings:                                                                            
   • asym_empir_1                                                                                                                 
   • asym_empir_2                                                                                                                 
   • asym_empir_3                                                                                                                 
   • asym_empir_4                                                                                                                 


⚠️ Switching peak profile type resets these settings to defaults:                                                                 
   • broad_gauss_u: 0.153402 -> 0.01                                                                                              
   • broad_gauss_v: -0.453103 -> -0.01                                                                                            
   • broad_gauss_w: 0.419409 -> 0.02                                                                                              
   • broad_lorentz_y: 0.086818 -> 0.0                                                                                             


Peak profile type for experiment 'pbso4' changed to


pseudo-voigt


## Fit edi-crysfml to FullProf

In [10]:
experiment.linked_structures['pbso4'].scale.free = True
experiment.instrument.calib_twotheta_offset.free = True

project.analysis.fit()
project.display.fit.results()

project.analysis.calculate()
calc_ed_crysfml_refined = experiment.data.intensity_calc

project.display.pattern_comparison(
    'pbso4',
    reference=calc_fullprof,
    candidate=calc_ed_crysfml_refined,
    reference_label='FullProf',
    candidate_label='edi-crysfml (refined)',
)

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'pbso4' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


...

,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,lmfit (leastsq)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),4.56
4,🔁 Iterations,89
5,📏 Goodness-of-fit (reduced χ²),214.95
6,"📏 R-factor (Rf, %)",4.75
7,"📏 R-factor squared (Rf², %)",4.91
8,"📏 Weighted R-factor (wR, %)",4.91


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,pbso4,linked_structure,pbso4,scale,,1.4638,1.4733,0.0013,0.65 % ↑
2,pbso4,instrument,,twotheta_offset,deg,-0.0842,-0.1285,0.0036,52.58 % ↑


## Agreement check

In [11]:
verify.assert_patterns_agree(
    [
        ('cryspy vs FullProf', calc_fullprof, calc_ed_cryspy_refined),
        ('crysfml vs FullProf', calc_fullprof, calc_ed_crysfml_refined),
    ],
    raise_on_failure=False,
)

,Comparison,Metric,Expected,Actual,OK
1,cryspy vs FullProf,Profile diff (%),< 2.5,0.91,✅
2,,Max deviation (%),< 6,5.57,✅
3,,Area ratio,0.99 to 1.01,1.0025,✅
4,,Shape correlation,> 0.999,0.9999,✅
5,crysfml vs FullProf,Profile diff (%),< 2.5,4.91,❌
6,,Max deviation (%),< 6,5.18,✅
7,,Area ratio,0.99 to 1.01,1.0023,✅
8,,Shape correlation,> 0.999,0.9983,❌


False